# RAGAS Evaluation — Faithfulness and Answer Relevancy
## A Hands-On Workshop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/74-ragas-evaluation/ragas_evaluation_workbook.ipynb)

Evaluates a RAG pipeline using RAGAS metrics: faithfulness (does the answer stick to the retrieved context?) and answer_relevancy (does it address the question?). Scores a 5-question golden dataset.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Why RAG metrics differ from accuracy; faithfulness vs relevancy |
| 2 | **Golden Dataset** — 5-row QA dataset with ground_truth and contexts |
| 3 | **RAG Pipeline** — Chroma retriever + LLM generates answers for each question |
| 4 | **RAGAS Evaluate** — `evaluate(dataset, metrics=[faithfulness, answer_relevancy])` |
| 5 | **Score Analysis** — Interpret scores; identify weak questions |
| ★ | **Exercises + Answer Key** |

### Prerequisites
- Python 3.10+, or Google Colab (install cell below)
- `OPENAI_API_KEY` in `.env` or Colab Secrets

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
        "langchain", "langchain-openai", "langchain-chroma", "chromadb", "ragas", "datasets", "python-dotenv"])
    print("Colab install complete.")
else:
    print("Local — skipping install.")

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy

DOCS = [
    Document(page_content="LangGraph is a library for building stateful multi-actor applications with LLMs."),
    Document(page_content="RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall."),
    Document(page_content="ChromaDB is an open-source embedding database for AI applications."),
    Document(page_content="Retrieval-augmented generation combines vector search with language model generation."),
    Document(page_content="LangSmith provides tracing and evaluation for LangChain and LangGraph applications."),
]

QA_ROWS = [
    {"question": "What is LangGraph?", "ground_truth": "LangGraph is a library for building stateful multi-actor applications with LLMs.", "contexts": ["LangGraph is a library for building stateful multi-actor applications with LLMs."]},
    {"question": "What does RAGAS evaluate?", "ground_truth": "RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall.", "contexts": ["RAGAS evaluates RAG pipelines on faithfulness, answer relevancy, and context recall."]},
    {"question": "What is ChromaDB?", "ground_truth": "ChromaDB is an open-source embedding database for AI applications.", "contexts": ["ChromaDB is an open-source embedding database for AI applications."]},
    {"question": "What is retrieval-augmented generation?", "ground_truth": "RAG combines vector search with language model generation.", "contexts": ["Retrieval-augmented generation combines vector search with language model generation."]},
    {"question": "What is LangSmith used for?", "ground_truth": "LangSmith provides tracing and evaluation for LangChain applications.", "contexts": ["LangSmith provides tracing and evaluation for LangChain and LangGraph applications."]},
]

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
vs = Chroma.from_documents(DOCS, embeddings, collection_name="ragas-eval")
retriever = vs.as_retriever(search_kwargs={"k": 1})

answers = []
for row in QA_ROWS:
    docs = retriever.invoke(row["question"])
    ctx = docs[0].page_content if docs else ""
    prompt = f"Context: {ctx}\n\nQuestion: {row['question']}\nAnswer briefly:"
    answer = llm.invoke([HumanMessage(content=prompt)]).content.strip()
    answers.append(answer)
    print(f"Q: {row['question']}")
    print(f"A: {answer}")

dataset = Dataset.from_list([
    {"question": r["question"], "answer": answers[i], "contexts": r["contexts"], "ground_truth": r["ground_truth"]}
    for i, r in enumerate(QA_ROWS)
])

print("\nRunning RAGAS evaluation...")
result = evaluate(dataset, metrics=[faithfulness, answer_relevancy])
print(f"\nRAGAS Scores:")
print(f"  Faithfulness:     {result['faithfulness']:.3f}")
print(f"  Answer Relevancy: {result['answer_relevancy']:.3f}")
print("\nInterpretation:")
print("  Faithfulness > 0.8 = answers grounded in context")
print("  Answer Relevancy > 0.8 = answers address the question")

---
*Workshop complete. Pausing — confirm to continue to the next notebook.*